# 02 · Insights de Negocio
**TUMIPAY — Prueba Técnica Data Science Engineer**

Este notebook responde explícitamente las 4 preguntas de negocio del enunciado, con métricas concretas que soportan cada hallazgo.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
pd.set_option('display.float_format', '{:,.2f}'.format)

DATA = '../data/'
df   = pd.read_csv(DATA + 'dataset_con_scores.csv')
pagos = pagos = pd.read_csv('../data/pagos.csv')
pagos['fecha_pago'] = pd.to_datetime(pagos['fecha_pago'])
pagos['fecha_vencimiento'] = pd.to_datetime(pagos['fecha_vencimiento'])
pagos_clean = pagos.sort_values('valor_pagado', ascending=False).drop_duplicates(['credito_id','numero_cuota'], keep='first')

print(f'Dataset: {df.shape} | Tasa mora: {df["mora_30"].mean():.2%}')

Dataset: (1411, 50) | Tasa mora: 23.95%


---
## Pregunta 1: ¿Qué factores están asociados a mayor mora?

Analizamos diferencias entre grupos usando medianas, medias y tests estadísticos.

In [2]:
from scipy import stats

factores = {}
variables = ['score_interno_originacion','score_externo','relacion_cuota_ingreso',
             'monto_credito','tasa_interes_mensual','ingreso_mensual_estimado','numero_dependientes']

results = []
for v in variables:
    g0 = df[df.mora_30==0][v].dropna()
    g1 = df[df.mora_30==1][v].dropna()
    t, p = stats.mannwhitneyu(g0, g1, alternative='two-sided')
    results.append({
        'Variable': v,
        'Media sin mora': g0.mean(),
        'Media con mora': g1.mean(),
        'Diferencia %': (g1.mean()-g0.mean())/g0.mean()*100,
        'p-valor': p,
        'Significativo': 'SÍ' if p < 0.05 else 'NO'
    })

results_df = pd.DataFrame(results).sort_values('Diferencia %', key=abs, ascending=False)
print(results_df.to_string(index=False))

                 Variable  Media sin mora  Media con mora  Diferencia %  p-valor Significativo
   relacion_cuota_ingreso            0.13            0.20         48.87     0.00            SÍ
 ingreso_mensual_estimado    2,965,321.53    2,587,781.05        -12.73     0.00            SÍ
            score_externo          603.81          548.85         -9.10     0.00            SÍ
score_interno_originacion          827.86          789.91         -4.58     0.00            SÍ
      numero_dependientes            1.26            1.30          3.78     0.38            NO
     tasa_interes_mensual            0.03            0.03          3.07     0.43            NO
            monto_credito    2,558,154.71    2,502,514.79         -2.18     0.28            NO


In [3]:
# Hallazgos clave — con evidencia
print('=== HALLAZGO 1: Score interno ===')
s0 = df[df.mora_30==0]['score_interno_originacion'].mean()
s1 = df[df.mora_30==1]['score_interno_originacion'].mean()
print(f'  Sin mora: {s0:.1f} puntos | Con mora: {s1:.1f} puntos')
print(f'  Diferencia: {s0-s1:.1f} puntos ({(s0-s1)/s0:.1%})')
print(f'  -> Un score interno bajo es el predictor más importante en originación.')

print('\n=== HALLAZGO 2: Relación Cuota/Ingreso ===')
r0 = df[df.mora_30==0]['relacion_cuota_ingreso'].mean()
r1 = df[df.mora_30==1]['relacion_cuota_ingreso'].mean()
print(f'  Sin mora: {r0:.3f} | Con mora: {r1:.3f}')
print(f'  Diferencia: +{(r1-r0)/r0:.1%}')
print(f'  -> Clientes con RCI > 0.35 tienen tasa de mora {df[df.relacion_cuota_ingreso>0.35]["mora_30"].mean():.1%} vs {df[df.relacion_cuota_ingreso<=0.35]["mora_30"].mean():.1%} en los demás.')

print('\n=== HALLAZGO 3: Ingreso mensual ===')
i0 = df[df.mora_30==0]['ingreso_mensual_estimado'].mean()
i1 = df[df.mora_30==1]['ingreso_mensual_estimado'].mean()
print(f'  Sin mora: ${i0:,.0f} | Con mora: ${i1:,.0f}')
print(f'  Diferencia: -{(i0-i1)/i0:.1%} de ingreso promedio')

=== HALLAZGO 1: Score interno ===
  Sin mora: 827.9 puntos | Con mora: 789.9 puntos
  Diferencia: 38.0 puntos (4.6%)
  -> Un score interno bajo es el predictor más importante en originación.

=== HALLAZGO 2: Relación Cuota/Ingreso ===
  Sin mora: 0.133 | Con mora: 0.198
  Diferencia: +48.9%
  -> Clientes con RCI > 0.35 tienen tasa de mora 40.4% vs 22.6% en los demás.

=== HALLAZGO 3: Ingreso mensual ===
  Sin mora: $2,965,322 | Con mora: $2,587,781
  Diferencia: -12.7% de ingreso promedio


---
## Pregunta 2: ¿Qué segmentos presentan mayor riesgo?


In [4]:
# Canal x Producto — segmentos críticos
pivot_cp = df.groupby(['canal_originacion','producto_credito'])['mora_30'].agg(['mean','count']).reset_index()
pivot_cp.columns = ['Canal','Producto','Tasa mora','N']
pivot_cp = pivot_cp[pivot_cp['N'] >= 20].sort_values('Tasa mora', ascending=False)
print('TOP 10 segmentos de mayor riesgo (Canal x Producto):')
print(pivot_cp.head(10).to_string(index=False))

print('\n--- SEGMENTO CRÍTICO ---')
row = pivot_cp.iloc[0]
print(f"Canal {row['Canal']} + {row['Producto']}: tasa de mora {row['Tasa mora']:.1%} (n={row['N']:.0f})")

TOP 10 segmentos de mayor riesgo (Canal x Producto):
      Canal                Producto  Tasa mora   N
     Aliado            Microcrédito       0.40  63
        Web Crédito libre inversión       0.37  65
     Aliado Crédito libre inversión       0.31  71
        Web        Avance de nómina       0.28  25
        Web            Microcrédito       0.28  50
        App Crédito libre inversión       0.27 196
        App            Microcrédito       0.27 146
Call center            Microcrédito       0.26  27
        App        Avance de nómina       0.23  77
     Aliado Crédito consumo digital       0.22 131

--- SEGMENTO CRÍTICO ---
Canal Aliado + Microcrédito: tasa de mora 39.7% (n=63)


In [5]:
# Demográfico: estrato + ocupación
print('Mora por estrato:')
mora_est = df.groupby('estrato')['mora_30'].agg(['mean','count'])
mora_est.columns = ['tasa_mora','n']
print(mora_est.sort_values('tasa_mora', ascending=False).to_string())

print('\nMora por ocupación:')
mora_ocu = df.groupby('ocupacion')['mora_30'].agg(['mean','count'])
mora_ocu.columns = ['tasa_mora','n']
print(mora_ocu.sort_values('tasa_mora', ascending=False).to_string())

Mora por estrato:
         tasa_mora    n
estrato                
1             0.30  197
2             0.26  407
3             0.23  439
4             0.22  243
5             0.14   95
6             0.13   30

Mora por ocupación:
                      tasa_mora    n
ocupacion                           
Desempleado                0.52   60
Independiente              0.35  246
Conductor/Repartidor       0.33  133
Estudiante                 0.31   58
Pensionado                 0.25   61
Comerciante                0.23  164
Empleado                   0.16  548
Microempresario            0.14  141


---
## Pregunta 3: ¿Qué patrones se observan en el comportamiento de pago?


In [6]:
evol = pagos_clean[pagos_clean['numero_cuota'] <= 12].groupby('numero_cuota').agg(
    pct_mora30   = ('dias_mora', lambda x: (x > 30).mean()),
    pct_puntual  = ('estado_pago', lambda x: (x=='Pagado').mean()),
    pct_parcial  = ('estado_pago', lambda x: (x=='Parcial').mean()),
    mora_media   = ('dias_mora', 'mean'),
    n            = ('pago_id', 'count'),
).reset_index()

print('Evolución por cuota:')
print(evol[['numero_cuota','pct_mora30','pct_puntual','pct_parcial','mora_media','n']]
      .to_string(index=False, float_format='{:.3f}'.format))

print(f'\nHallazgos:')
print(f'  - La cuota 1 muestra la mayor mora media: {evol[evol.numero_cuota==1]["mora_media"].values[0]:.1f} días')
print(f'  - El % de pago puntual es relativamente estable: {evol["pct_puntual"].mean():.1%} promedio')
print(f'  - Los pagos parciales son bajos pero constantes: {evol["pct_parcial"].mean():.1%} promedio')
print(f'  - La mora no escala exponencialmente por cuota -> patrón difuso, no de deterioro progresivo claro')

Evolución por cuota:
 numero_cuota  pct_mora30  pct_puntual  pct_parcial  mora_media    n
            1       0.060        0.806        0.024       5.147 1527
            2       0.048        0.787        0.022       4.519 1480
            3       0.054        0.802        0.016       4.874 1411
            4       0.048        0.793        0.024       4.146 1154
            5       0.056        0.781        0.021       5.030 1058
            6       0.059        0.783        0.010       4.787  985
            7       0.056        0.769        0.012       4.704  588
            8       0.051        0.796        0.015       4.267  529
            9       0.051        0.771        0.008       4.129  472
           10       0.036        0.783        0.011       3.895  276
           11       0.052        0.823        0.008       4.060  248
           12       0.053        0.796        0.027       4.429  226

Hallazgos:
  - La cuota 1 muestra la mayor mora media: 5.1 días
  - El % de pago 

In [7]:
# Comportamiento digital previo a mora
print('Comportamiento digital por grupo de mora:')
dig = df.groupby('mora_30')[['pagos_fallidos','pct_exitosos','solicitudes_soporte',
                               'tasa_conversion_pago','total_eventos']].mean()
dig.index = ['Sin mora','Con mora']
print(dig.T.to_string())

Comportamiento digital por grupo de mora:
                      Sin mora  Con mora
pagos_fallidos            0.47      0.42
pct_exitosos              0.76      0.74
solicitudes_soporte       1.03      1.05
tasa_conversion_pago      0.73      0.77
total_eventos            12.27     12.01


---
## Pregunta 4: Recomendaciones accionables


In [8]:
print('=== PARA RIESGO ===')
print()
print('1. POLÍTICA DE SCORE MÍNIMO:')
print(f'   Clientes con score_interno < 790 tienen tasa de mora {df[df.score_interno_originacion < 790]["mora_30"].mean():.1%}')
print(f'   vs {df[df.score_interno_originacion >= 790]["mora_30"].mean():.1%} para score >= 790.')
print(f'   Recomendación: revisar el umbral de corte de score interno en originación.')
print()
print('2. POLÍTICA DE RELACIÓN CUOTA/INGRESO:')
print(f'   RCI > 0.35: mora {df[df.relacion_cuota_ingreso>0.35]["mora_30"].mean():.1%}')
print(f'   RCI <= 0.35: mora {df[df.relacion_cuota_ingreso<=0.35]["mora_30"].mean():.1%}')
print(f'   Recomendación: aplicar límite duro de RCI <= 0.35 en aprobación.')
print()
print('=== PARA PRODUCTO ===')
print()
print('3. CANAL ALIADO + MICROCRÉDITO:')
crit = df[(df.canal_originacion=='Aliado')&(df.producto_credito=='Microcrédito')]
print(f'   Tasa de mora: {crit["mora_30"].mean():.1%} (n={len(crit)})')
print(f'   Recomendación: revisar controles de originación en canal Aliado.')
print(f'   Evaluar incentivos del aliado que podrían favorecer colocación sobre calidad.')
print()
print('=== PARA OPERACIONES ===')
print()
print('4. GESTIÓN DE COBRANZA TEMPRANA:')
print(f'   La mora se manifiesta desde la cuota 1. No hay período de gracia natural.')
print(f'   Recomendación: activar recordatorio de pago preventivo antes del vencimiento de cuota 1.')
print()
print('5. SEÑALES DIGITALES DE ALERTA:')
dig_mora = df[df.mora_30==1]
print(f'   Clientes con mora registran {dig_mora["pagos_fallidos"].mean():.1f} pagos fallidos en app vs {df[df.mora_30==0]["pagos_fallidos"].mean():.1f} sin mora.')
print(f'   Recomendación: usar pagos_fallidos como señal de alerta temprana en cobranza.')

=== PARA RIESGO ===

1. POLÍTICA DE SCORE MÍNIMO:
   Clientes con score_interno < 790 tienen tasa de mora 38.6%
   vs 17.9% para score >= 790.
   Recomendación: revisar el umbral de corte de score interno en originación.

2. POLÍTICA DE RELACIÓN CUOTA/INGRESO:
   RCI > 0.35: mora 40.4%
   RCI <= 0.35: mora 22.6%
   Recomendación: aplicar límite duro de RCI <= 0.35 en aprobación.

=== PARA PRODUCTO ===

3. CANAL ALIADO + MICROCRÉDITO:
   Tasa de mora: 39.7% (n=63)
   Recomendación: revisar controles de originación en canal Aliado.
   Evaluar incentivos del aliado que podrían favorecer colocación sobre calidad.

=== PARA OPERACIONES ===

4. GESTIÓN DE COBRANZA TEMPRANA:
   La mora se manifiesta desde la cuota 1. No hay período de gracia natural.
   Recomendación: activar recordatorio de pago preventivo antes del vencimiento de cuota 1.

5. SEÑALES DIGITALES DE ALERTA:
   Clientes con mora registran 0.4 pagos fallidos en app vs 0.5 sin mora.
   Recomendación: usar pagos_fallidos como seña